# 01 — Dataset preparation

Builds the metadata table for the WikiArt collection and selects the 20 painting movements used in the paper.

**Input** (Kaggle copy of WikiArt, `steubk/wikiart`), placed under `data/wikiart/`:
- one folder per style/movement containing the images,
- `classes.csv` (file name, artist, genre/style list, phash, …) and `wclasses.csv`.

**Output**
- `data/metadata/artemis_metadata.parquet` — metadata for all 80,042 records,
- `data/metadata/selected_movements.json` — the 20 selected movements,
- `data/metadata/image_list.csv.gz` — the 65,998 paintings used in the paper,
- summary tables in `results/excel/`.

The images themselves are not redistributed with this repository.

In [ ]:
import json
import warnings
from difflib import get_close_matches, SequenceMatcher

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")

In [ ]:
import os
from pathlib import Path

# Project root: the repository folder (notebooks/ is one level below it).
# Override with the environment variable ARTEMIS_ROOT if the data live elsewhere.
PROJECT_ROOT = Path(os.environ.get("ARTEMIS_ROOT", Path.cwd().parent)).resolve()

DATA_DIR       = PROJECT_ROOT / "data"
WIKIART_DIR    = DATA_DIR / "wikiart"
METADATA_DIR   = DATA_DIR / "metadata"
RESULTS_DIR    = PROJECT_ROOT / "results"
EXCEL_DIR      = RESULTS_DIR / "excel"

for folder in (DATA_DIR, METADATA_DIR, RESULTS_DIR, EXCEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "WikiArt"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("WIKIART_DIR  :", WIKIART_DIR)
print("METADATA_DIR :", METADATA_DIR)

## Scan the movement folders

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

movement_folders = sorted(f for f in WIKIART_DIR.iterdir() if f.is_dir())
print(f"Movement folders: {len(movement_folders)}\n")

folder_counts, total_images = [], 0
for folder in movement_folders:
    n = sum(1 for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS)
    total_images += n
    folder_counts.append({"movement": folder.name, "image_count": n})
    print(f"{folder.name:30s} : {n:6d}")

print(f"\nTotal images on disk: {total_images:,}")

## Read the WikiArt metadata files

In [ ]:
CLASSES_CSV  = WIKIART_DIR / "classes.csv"
WCLASSES_CSV = WIKIART_DIR / "wclasses.csv"
assert CLASSES_CSV.exists() and WCLASSES_CSV.exists(), "classes.csv / wclasses.csv not found in data/wikiart"

classes_df  = pd.read_csv(CLASSES_CSV)
wclasses_df = pd.read_csv(WCLASSES_CSV)
print(f"classes.csv  : {classes_df.shape[0]:,} rows, {classes_df.shape[1]} columns")
print(f"wclasses.csv : {wclasses_df.shape[0]:,} rows, {wclasses_df.shape[1]} columns")

## Build the metadata table

The movement of each record is taken from the first component of its file path
(e.g. `Impressionism/claude-monet_water-lilies.jpg` → `Impressionism`).

In [ ]:
artemis_df = classes_df.copy()
artemis_df.insert(0, "image_id", [f"ART{i:06d}" for i in range(1, len(artemis_df) + 1)])
artemis_df["movement"] = artemis_df["filename"].str.split("/").str[0]
artemis_df["title"] = artemis_df["description"]
artemis_df["dataset"] = DATASET_NAME

priority = ["image_id", "filename", "artist", "title", "dataset", "subset", "width", "height"]
rest = [c for c in artemis_df.columns if c not in priority + ["movement"]]
artemis_df = artemis_df[priority + rest + ["movement"]]

print(f"Records: {len(artemis_df):,}")
print("Columns:", list(artemis_df.columns))
artemis_df.head()

## Synchronise file names with the files on disk

Some file names in `classes.csv` differ from the names on disk because of character-encoding
issues. Each missing name is matched to the closest file name in the same movement folder
(`difflib.get_close_matches`, cutoff 0.70). Every correction is logged to
`results/excel/filename_sync_log.xlsx`, and records mapped to the same file are reported.

In [ ]:
real_files = {}
for movement_dir in WIKIART_DIR.iterdir():
    if movement_dir.is_dir():
        real_files[movement_dir.name] = {f.name.lower(): f.name for f in movement_dir.glob("*") if f.is_file()}

fixed, not_found, sync_log = 0, 0, []
for idx, row in artemis_df.iterrows():
    movement = row["movement"]
    filename = Path(row["filename"]).name
    folder = Path(row["filename"]).parent.as_posix()
    if (WIKIART_DIR / row["filename"]).exists():
        continue
    match = get_close_matches(filename.lower(), list(real_files[movement].keys()), n=1, cutoff=0.70)
    if not match:
        not_found += 1
        continue
    new_filename = real_files[movement][match[0]]
    artemis_df.at[idx, "filename"] = folder + "/" + new_filename
    sync_log.append({"image_id": row["image_id"], "old_filename": row["filename"],
                     "new_filename": folder + "/" + new_filename,
                     "match_ratio": round(SequenceMatcher(None, filename.lower(), match[0]).ratio(), 4)})
    fixed += 1

print(f"Corrected file names : {fixed}")
print(f"Not found            : {not_found}")
if sync_log:
    pd.DataFrame(sync_log).to_excel(EXCEL_DIR / "filename_sync_log.xlsx", index=False)
print("Records mapped to the same file:", int(artemis_df["filename"].duplicated(keep=False).sum()))

## Movement and artist summaries (all 27 WikiArt styles)

In [ ]:
movement_summary = (artemis_df.groupby("movement")
                    .agg(Image_Count=("image_id", "count"), Artist_Count=("artist", "nunique"),
                         Avg_Width=("width", "mean"), Avg_Height=("height", "mean"))
                    .sort_values("Image_Count", ascending=False).reset_index())
movement_summary[["Avg_Width", "Avg_Height"]] = movement_summary[["Avg_Width", "Avg_Height"]].round(0).astype(int)

artist_summary = (artemis_df.groupby("artist")
                  .agg(Image_Count=("image_id", "count"), Movement_Count=("movement", "nunique"))
                  .sort_values("Image_Count", ascending=False).reset_index())

print(f"Movements: {len(movement_summary)}   Artists: {len(artist_summary):,}")
movement_summary

## Select the 20 movements used in the paper

The 20 movements cover European and North American painting from the late 18th to the 20th century.
Early, High and Northern Renaissance, Mannerism, Baroque, Rococo and Ukiyo-e are excluded.

In [ ]:
SELECTED_MOVEMENTS = [
    "Abstract_Expressionism", "Action_painting", "Analytical_Cubism", "Art_Nouveau_Modern",
    "Color_Field_Painting", "Contemporary_Realism", "Cubism", "Expressionism", "Fauvism",
    "Impressionism", "Minimalism", "Naive_Art_Primitivism", "New_Realism", "Pointillism",
    "Pop_Art", "Post_Impressionism", "Realism", "Romanticism", "Symbolism", "Synthetic_Cubism",
]

artemis_selected = artemis_df[artemis_df["movement"].isin(SELECTED_MOVEMENTS)].copy()
missing = sorted(set(SELECTED_MOVEMENTS) - set(artemis_selected["movement"]))
assert not missing, f"Movements not found: {missing}"

selected_summary = (artemis_selected.groupby("movement")
                    .agg(Image_Count=("image_id", "count"), Artist_Count=("artist", "nunique"))
                    .sort_values("Image_Count", ascending=False).reset_index())

print(f"Selected paintings : {len(artemis_selected):,}")    # 65,998
print(f"Movements          : {artemis_selected['movement'].nunique()}")
print(f"Artists            : {artemis_selected['artist'].nunique():,}")   # 936
selected_summary

In [ ]:
artist_per_movement_summary = (artemis_df.groupby(["movement", "artist"])
                               .agg(Image_Count=("image_id", "count"))
                               .sort_values(["movement", "Image_Count"], ascending=[True, False])
                               .reset_index())
artist_per_movement_summary.head(20)

## Save the metadata and summary tables

In [ ]:
artemis_df.to_parquet(METADATA_DIR / "artemis_metadata.parquet", index=False)

with open(METADATA_DIR / "selected_movements.json", "w", encoding="utf-8") as f:
    json.dump({"dataset": DATASET_NAME, "movement_count": len(SELECTED_MOVEMENTS),
               "image_count": int(len(artemis_selected)), "movements": sorted(SELECTED_MOVEMENTS)},
              f, indent=4, ensure_ascii=False)

movement_summary.to_excel(EXCEL_DIR / "movement_summary.xlsx", index=False)
artist_summary.to_excel(EXCEL_DIR / "artist_summary.xlsx", index=False)
selected_summary.to_excel(EXCEL_DIR / "selected_movements_summary.xlsx", index=False)
artist_per_movement_summary.to_excel(EXCEL_DIR / "artist_per_movement_summary.xlsx", index=False)

print("✓ artemis_metadata.parquet, selected_movements.json and summary tables saved.")

## Export the list of paintings used in the paper

`image_list.csv.gz` lists the 65,998 paintings of the 20 selected movements with their movement, artist
and WikiArt style list (`genre`). Together with the split files in `splits/`, it fully defines the data
used in every experiment.

In [ ]:
image_list = (artemis_selected[["filename", "movement", "artist", "genre"]]
              .reset_index(drop=True))
image_list.to_csv(METADATA_DIR / "image_list.csv.gz", index=False)
print(f"✓ image_list.csv.gz: {len(image_list):,} paintings")

## Check that every image referenced in the metadata exists on disk

In [ ]:
missing_df = pd.DataFrame([{"image_id": r.image_id, "filename": r.filename, "movement": r.movement}
                           for r in artemis_df.itertuples() if not (WIKIART_DIR / r.filename).exists()])
print(f"Missing files: {len(missing_df)}")
if len(missing_df):
    missing_df.to_excel(EXCEL_DIR / "missing_files.xlsx", index=False)